# Module 1: Agentic RAG

### Part 1: RAG

Building a working Retrieval-Augmented Generation (RAG) pipeline from scratch.

LLMs have limitations:
* Knowledge cutoff
* No access to your data
* Hallucinations

RAG solves the above limitations by giving LLMs relevant docs at question time. 

**Environment Setup:**

```
uv add requests minsearch openai jupyter python-dotenv 
```

In [6]:
from dotenv import load_dotenv
load_dotenv()

True

In [7]:
from openai import OpenAI
import os

# Point the OpenAI client to Google's base URL
client = OpenAI(
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key=os.environ.get("GEMINI_API_KEY")
)

In [8]:
def llm(prompt):
    response = client.chat.completions.create(
        model="gemini-2.5-flash",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content


In [9]:
# Blackbox test without the LLM having context
llm("Hey, what's up?")

"Hey there! Not much on my end, just here and ready to chat and help with whatever you need.\n\nWhat's up with you? Or what can I do for you today?"

In [10]:
# course specific question - LLm still does not have context
question = "I just discovered teh course. Can I join?"
answer = llm(question)
print(answer)

That's exciting! I'd love to help you figure out if you can join.

To give you the best information, I'll need a few more details. Could you please tell me:

1.  **What is the name of the course?** (This is the most crucial piece of information!)
2.  **Where did you discover it?** (e.g., a specific website, a university page, an online learning platform like Coursera/edX, a local institution, etc.)
3.  **Do you have a link to the course page?** (This would be ideal!)

Once I have that information, I can try to look up its enrollment status, registration deadlines, and any prerequisites or requirements.


### Adding more context manually

In [11]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [12]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [13]:
answer = llm(prompt)
print(answer)

Yes, you can still join. If you want to receive a certificate, you need to submit your project while submissions are still being accepted.


This is the answer we actually want to give to our students. What we just did is nothing but RAG.

### Retrieval plus generation

What we just did was naive. We knew in advance which FAQ entry held the answer and pasted it in by hand. What we want instead is to perform search automatically. We take the student's question, find the most relevant documents, and send those to the LLM.

In code, it looks like this:

```python
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)
```

It comes down to three components. The pieces are:
* search
* prompt
* LLM

The model is only as good as the retrieval, so search quality matters a lot for RAG.